# Retail Demand Forecasting with Time-Aware Regression

## tl;dr

This study compares transparent regression baselines with XGBoost, LightGBM, and a validation-weighted ensemble on 3.0 million retail sales records. The last 90 days are held out for final evaluation.

The ensemble achieves **MAE 29.4**, **RMSE 42.8**, and **R² 0.784**, reducing RMSE by **55.1%** relative to the lag-1 linear baseline. Its gain over LightGBM is only **1.2%**, so the operational case for maintaining two models remains weak without broader rolling backtests.


## Context & Methods

The target is company-level daily average sales. This grain is useful for testing aggregate demand signals, but it is not a substitute for store-family forecasting.

### Key assumptions

- Same-day promotion volume is known because promotions are planned before the sales date.
- Every sales-derived feature is shifted by at least one day.
- Validation data may guide early stopping and ensemble weights; test data may not.
- MAE and RMSE are proxy objectives because inventory-cost labels are unavailable.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from retail_forecasting import (
    build_daily_model_table,
    fit_boosting_models,
    load_sales,
    temporal_split,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.3f}".format

## Data

Download the data from the [Kaggle Store Sales competition](https://www.kaggle.com/competitions/store-sales-time-series-forecasting). The raw files are intentionally excluded from this repository.


In [ ]:
DATA_DIR = REPO_ROOT / "input" / "store-sales-time-series-forecasting"

sales = load_sales(DATA_DIR)
model_table = build_daily_model_table(sales)

print(f"Raw rows: {len(sales):,}")
print(f"Coverage: {sales['date'].min().date()} to {sales['date'].max().date()}")
print(f"Modeling rows after lag construction: {len(model_table):,}")

## Exploratory analysis

The plots below establish whether trend, weekly seasonality, and promotion exposure are material enough to include in the modeling table.


In [ ]:
daily = sales.groupby("date", as_index=False).agg(
    average_sales=("sales", "mean"),
    promotion_units=("onpromotion", "sum"),
)
daily["rolling_mean_28"] = daily["average_sales"].rolling(28, min_periods=7).mean()
daily["day_of_week"] = daily["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"
]
weekday_sales = (
    daily.groupby("day_of_week")["average_sales"]
    .mean()
    .reindex(weekday_order)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(daily["date"], daily["average_sales"], color="0.70", linewidth=0.8)
axes[0].plot(daily["date"], daily["rolling_mean_28"], color="#2F6B9A", linewidth=2)
axes[0].set(title="Daily Sales and 28-Day Moving Average", xlabel="Date", ylabel="Average sales")

axes[1].bar(weekday_sales.index, weekday_sales.values, color="#D98A24")
axes[1].set(title="Average Sales by Day of Week", xlabel="", ylabel="Average sales")
axes[1].tick_params(axis="x", rotation=35)
fig.tight_layout()

## Temporal validation

The split follows the forecasting timeline. The validation window selects the number of boosting rounds and ensemble weights; the final window is used only once for model comparison.


In [ ]:
train, validation, test = temporal_split(model_table)

split_summary = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "start": [train.index.min(), validation.index.min(), test.index.min()],
        "end": [train.index.max(), validation.index.max(), test.index.max()],
        "observations": [len(train), len(validation), len(test)],
    }
)
split_summary

## Results

The comparison includes two transparent baselines, two regularized gradient-boosting models, and a validation-weighted blend. The baseline models remain in the table because improvement is only meaningful relative to a credible alternative.


In [ ]:
metrics, predictions = fit_boosting_models(train, validation, test)
metrics.round(3)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

for column, color in {
    "actual": "#374151",
    "xgboost": "#4C78A8",
    "lightgbm": "#F2A104",
    "ensemble": "#C94B5F",
}.items():
    axes[0].plot(predictions.index, predictions[column], label=column.title(), color=color)

axes[0].set(title="Locked 90-Day Test: Actual vs Predictions", ylabel="Average sales")
axes[0].legend(frameon=False, ncol=4)

ranking = metrics.sort_values("rmse", ascending=True)
axes[1].barh(ranking["model"], ranking["rmse"], color="#4C78A8")
axes[1].set(title="Test RMSE by Model", xlabel="RMSE (lower is better)", ylabel="")
axes[1].bar_label(axes[1].containers[0], fmt="%.1f", padding=4)
fig.tight_layout()

## Takeaways

1. Calendar and lag structure contain substantially more signal than a one-day autoregressive baseline.
2. LightGBM captures nearly all of the ensemble's predictive value with lower operating complexity.
3. Residual spikes suggest omitted holiday, event, and local store-family effects.
4. The next evaluation should use rolling-origin backtests at `date × store × product_family` grain.
5. Feature importance is diagnostic rather than causal; promotion decisions require a separate identification strategy.

### 日本語要約

時系列の順序を維持した検証により、将来情報のリークを防止しています。アンサンブルは高い精度を示しましたが、LightGBM単体との差は小さいため、運用コストを含めた判断が必要です。次のステップは、店舗・商品カテゴリ単位への粒度拡張とローリング検証です。
